In [1]:
import re
from pydantic import BaseModel, Field
from typing import Optional
from langgraph.graph import StateGraph, END
from typing import Optional

In [2]:
# RAG Setup (Run Once)

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load rules
loader = TextLoader("loan_rules.txt")
documents = loader.load()

# Split
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.split_documents(documents)

# Create vector DB
vectorstore = FAISS.from_documents(docs, embeddings)

# Save index
vectorstore.save_local("loan_policy_index")

print("✅ RAG index created")

c:\Users\User\anaconda3\envs\rag_agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_7892\3712463605.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2942.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_

✅ RAG index created


In [3]:
def query_policies(question: str) -> str:
    db = FAISS.load_local(
        "loan_policy_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

    results = db.similarity_search(question, k=3)
    return "\n".join([doc.page_content for doc in results])

In [4]:
# 1. FIRST define LoanState
class LoanState(BaseModel):
    user_input: str
    salary: Optional[int] = None
    credit_score: Optional[int] = None
    loan_amount: Optional[int] = None
    tenure_years: Optional[int] = None
    existing_emi: Optional[int] = None
    retrieved_rules: Optional[str] = None
    eligibility_status: Optional[str] = None
    approval_score: Optional[float] = None
    suggested_plan: Optional[str] = None
    final_reply: Optional[str] = None

In [ ]:
# 2. THEN define functions
#def collect_financials(state: LoanState):
   # return state

# 2. THEN define Node functions

In [5]:


def collect_financials(state: LoanState):
    text = state.user_input.lower()

    salary = re.search(r'(\d+)\s?k', text)
    credit = re.search(r'credit score\s*(\d+)', text)
    loan = re.search(r'(\d+)\s?lakhs?', text)
    tenure = re.search(r'(\d+)\s?years?', text)
    emi = re.search(r'emi\s*(\d+)', text)

    if salary:
        state.salary = int(salary.group(1)) * 1000
    if credit:
        state.credit_score = int(credit.group(1))
    if loan:
        state.loan_amount = int(loan.group(1)) * 100000
    if tenure:
        state.tenure_years = int(tenure.group(1))
    if emi:
        state.existing_emi = int(emi.group(1)) * 1000

    return state

In [ ]:
#from build_rag import query_policies

In [6]:
def check_eligibility(state: LoanState):

    rules = query_policies("loan eligibility rules")
    state.retrieved_rules = rules

    # Basic checks
    if state.salary < 25000:
        state.eligibility_status = "Not Eligible (Low Salary)"
        return state

    if state.credit_score < 700:
        state.eligibility_status = "Not Eligible (Low Credit Score)"
        return state

    # DTI Calculation
    dti = (state.existing_emi / state.salary) * 100

    if dti > 55:
        state.eligibility_status = "Not Eligible (High DTI)"
    else:
        state.eligibility_status = "Eligible"

    return state

In [7]:
def predict_approval_chance(state: LoanState):

    score = 50

    if state.credit_score > 750:
        score += 20
    elif state.credit_score > 700:
        score += 10

    if state.salary > 50000:
        score += 10

    dti = (state.existing_emi / state.salary) * 100

    if dti < 30:
        score += 10
    elif dti > 50:
        score -= 10

    state.approval_score = min(score, 100)

    return state

In [8]:
def suggest_loan_plan(state: LoanState):

    P = state.loan_amount
    r = 0.10 / 12  # 10% yearly interest
    n = state.tenure_years * 12

    emi = int((P * r * (1+r)**n) / ((1+r)**n - 1))

    state.suggested_plan = f"""
Loan Amount: ₹{P}
Tenure: {state.tenure_years} years
Estimated EMI: ₹{emi}
"""

    return state

In [9]:
def review_response(state: LoanState):

    state.final_reply = f"""
Loan Eligibility: {state.eligibility_status}

Approval Chance: {state.approval_score}%

Recommended Plan:
{state.suggested_plan}

Suggestion:
- Maintain good credit score
- Avoid high EMI burden
- Ensure stable income
"""

    return state

# 3. build the workflow graph

In [10]:
workflow = StateGraph(LoanState)

In [11]:
workflow.add_node("collect_financials", collect_financials)
workflow.add_node("check_eligibility", check_eligibility)
workflow.add_node("predict_approval_chance", predict_approval_chance)
workflow.add_node("suggest_loan_plan", suggest_loan_plan)
workflow.add_node("review_response", review_response)

In [12]:
workflow.set_entry_point("collect_financials")

In [13]:
workflow.add_edge("collect_financials", "check_eligibility")
workflow.add_edge("check_eligibility", "predict_approval_chance")
workflow.add_edge("predict_approval_chance", "suggest_loan_plan")
workflow.add_edge("suggest_loan_plan", "review_response")
workflow.add_edge("review_response", END)

In [14]:
loan_app = workflow.compile()

# 4. THEN run the workflow by taking user query

In [15]:
def ask_loan_ai(user_query: str) -> str:
    try:
        result = loan_app.invoke({"user_input": user_query})
        return result.get("final_reply", "No response generated.")
    except Exception as e:
        return f"Error: {str(e)}"

In [17]:
# -------- CLI LOOP --------

if __name__ == "__main__":
    print("💬 Loan Eligibility Assistant (type 'exit' to quit)\n")

    while True:
        user_query = input("Enter your details: ")

        if user_query.lower() in ["exit", "quit"]:
            print("👋 Exiting...")
            break

        response = ask_loan_ai(user_query)
        print("\n🤖 Response:\n", response)
        print("-" * 50)

💬 Loan Eligibility Assistant (type 'exit' to quit)


🤖 Response:
 
Loan Eligibility: Eligible

Approval Chance: 80.0%

Recommended Plan:

Loan Amount: ₹1000000
Tenure: 3 years
Estimated EMI: ₹32267


Suggestion:
- Maintain good credit score
- Avoid high EMI burden
- Ensure stable income

--------------------------------------------------
👋 Exiting...


In [ ]:
# I earn 65k/month, credit score 740, loan required 10 lakhs for 3 years. Already paying emi 12k.